# Folded-Clos Topology Visualization Example

This notebook demonstrates how to generate and visualize a single Folded-Clos topology with a 3-level hierarchy (Server -> Nodes -> NPUs).

In [ ]:
import sys
sys.path.insert(0, '/app/astra-sim/optimos/create_topology/new')

import importlib
import matplotlib.pyplot as plt

# Reload modules to pick up any changes
import create_topology
importlib.reload(create_topology)
import utils
importlib.reload(utils)

from create_topology import FoldedClos, CustomizedDragonfly
from utils import visualize_topology, visualize_single_server
from create_topology_main import generate_topology_files

## 1. Define Topology Configuration

Here, we define the configuration for a K=4 Folded-Clos (Fat-Tree) with a 3-level hierarchy:
- **16 Servers**: The base fat-tree has 16 servers.
- **2 Nodes per Server**: Each server is divided into 2 logical nodes.
- **4 NPUs per Node**: Each node contains 4 NPUs.
- **Total NPUs**: 16 * 2 * 4 = 128 NPUs.

In [ ]:
# Define bandwidth configuration for the topology
bw_config = {
    'host_edge': 25,         # Host (ToR) to edge switch
    'edge_agg': 25,          # Edge to aggregation switch
    'agg_core': 25,          # Aggregation to core switch
    'intra_node': 25         # Within a node (NPU to NPU-switch)
}

latency_config = {
    'host_edge': 0.001,      # Host to edge switch latency
    'edge_agg': 0.01,        # Edge to aggregation latency
    'agg_core': 0.05,        # Aggregation to core latency
    'intra_node': 0.0001     # Within node latency
}

# Create an instance of the FoldedClos topology
config = {
    'K': 4,
    'bandwidth_config': bw_config,
    'latency_config': latency_config,
    'bw_unit': 'GB/s',
    'lat_unit': 'ms',
    'npus_per_node': 1,
    'intra_node_topology': 'switch',
    'num_nvswitches': 1  # Using 2 NVSwitches per node for redundancy/performance
}
fc_3level = FoldedClos(
    K=4,                      # K=4 fat-tree results in 16 servers
    bandwidth_config=bw_config,
    latency_config=latency_config,
    npus_per_node=1,          # 4 NPUs per node
    intra_node_topology='switch',
    # num_nvswitches=1
)

print("Folded-Clos Configuration:")
print(f"  - K: {fc_3level.K}")
print(f"  - Total servers: {fc_3level.total_num_servers}")
print(f"  - Nodes per server: {fc_3level.nodes_per_server}")
print(f"  - Total nodes (sub-groups): {fc_3level.total_num_nodes}")
print(f"  - NPUs per node: {fc_3level.npus_per_node}")
print(f"  - Total NPUs: {fc_3level.total_num_hosts}")

generate_topology_files('FoldedClos', 'ECMP', config, '.', 'FoldedClosECMP')
generate_topology_files('FoldedClos', 'Uniform', config, '.', 'FoldedClosUniform')
generate_topology_files('FoldedClos', 'Random', config, '.', 'FoldedClosDet_v1')

## 2. Visualize the Full Topology

This visualization shows the entire 128-NPU network. It can be complex, but it illustrates the overall structure with Core, Aggregation, and Edge switches.

In [ ]:
# Visualize the entire topology
# Note: show_bw=False is used to keep the visualization clean for large graphs
fig, ax, G = visualize_topology(
    fc_3level, 
    "Folded-Clos (K=4) with 3-Level Hierarchy", 
    figsize=(20, 10), 
    show_bw=True
)
plt.show()

In [ ]:
# Dragonfly with 1 NPU per switch
bw_config = {
    'host_switch': 25,
    'intra_group': 25,
    'inter_group': 25,
    'intra_node': 25 # Not relevant with 1 NPU
}

latency_config = {
    'host_switch': 0.001,
    'intra_group': 0.01,
    'inter_group': 0.1,
    'intra_node': 0.0001
}

config = {
    'G': 4,
    'A': 4,
    'h': 2,
    'concentration': 1,          # 1 server per switch
    'nodes_per_server': 1,       # 1 node per server
    'bandwidth_config': bw_config,
    'latency_config': latency_config,
    'bw_unit': 'GB/s',
    'lat_unit': 'ms',
    'npus_per_node': 1,
    'intra_node_topology': 'switch',
}
df_one_npu = CustomizedDragonfly(
    G=4,
    A=4,
    h=2,
    concentration=1,          # 1 server per switch
    nodes_per_server=1,       # 1 node per server
    bandwidth_config=bw_config,
    latency_config=latency_config,
    npus_per_node=1,          # 1 NPU per node
    intra_node_topology='switch' # Not relevant with 1 NPU
)

fig, ax, G = visualize_topology(df_one_npu, "Dragonfly with 1 NPU per Switch")
plt.show()

generate_topology_files('Dragonfly', 'ECMP', config, '.', 'DragonflyECMP')
generate_topology_files('Dragonfly', 'Uniform', config, '.', 'DragonflyUniform')
generate_topology_files('Dragonfly', 'Random', config, '.', 'DragonflyDet_v1')

## 3. Folded-Clos with 8 GPUs per Node (128 NPUs total)

This example creates a Folded-Clos (K=4) topology with 128 NPUs, where each of the 16 servers contains one node with 8 GPUs.
- **K=4**: 16 servers
- **Nodes per server**: 1
- **NPUs per node**: 8
- **Total NPUs**: 16 * 1 * 8 = 128
- **Latency**: 200ns for intra-node links and 500ns for all other links.

In [ ]:
# Define bandwidth configuration for the topology
bw_config = {
    'host_edge': 50,         # Host (ToR) to edge switch
    'edge_agg': 50,          # Edge to aggregation switch
    'agg_core': 50,          # Aggregation to core switch
    'intra_node': 600         # Within a node (NPU to NPU-switch)
}

# Latency: 200ns for intra-node, 500ns for other links
# 200 ns = 0.2 us = 0.0002 ms
# 500 ns = 0.5 us = 0.0005 ms
latency_config = {
    'host_edge': 0.0005,
    'edge_agg': 0.0005,
    'agg_core': 0.0005,
    'intra_node': 0.0002
}

# Create an instance of the FoldedClos topology
config = {
    'K': 4,
    'bandwidth_config': bw_config,
    'latency_config': latency_config,
    'bw_unit': 'GB/s',
    'lat_unit': 'ms',
    'nodes_per_server': 1,
    'npus_per_node': 8,
    'intra_node_topology': 'switch',
    'num_nvswitches': 1
}

fc_8gpu = FoldedClos(
    K=4,
    bandwidth_config=bw_config,
    latency_config=latency_config,
    nodes_per_server=1,
    npus_per_node=8,
    intra_node_topology='switch',
    num_nvswitches=1
)

print("Folded-Clos Configuration (8 GPUs/Node):")
print(f"  - K: {fc_8gpu.K}")
print(f"  - Total servers: {fc_8gpu.total_num_servers}")
print(f"  - Nodes per server: {fc_8gpu.nodes_per_server}")
print(f"  - Total nodes: {fc_8gpu.total_num_nodes}")
print(f"  - NPUs per node: {fc_8gpu.npus_per_node}")
print(f"  - Total NPUs: {fc_8gpu.total_num_hosts}")

# Visualize the topology
fig, ax, G = visualize_topology(
    fc_8gpu, 
    "Folded-Clos (K=4) with 8 GPUs/Node (128 NPUs)", 
    figsize=(20, 10), 
    show_bw=True
)
plt.show()

# Generate topology files
generate_topology_files('FoldedClos', 'ECMP', config, '.', 'FoldedClos_8GPU_ECMP')
# generate_topology_files('FoldedClos', 'Uniform', config, '.', 'FoldedClos_8GPU_Uniform')
# generate_topology_files('FoldedClos', 'Random', config, '.', 'FoldedClos_8GPU_Det_v1')


In [ ]:
# Define bandwidth configuration for the topology
bw_config = {
    'host_edge': 50,         # Host (ToR) to edge switch
    'edge_agg': 50,          # Edge to aggregation switch
    'agg_core': 50,          # Aggregation to core switch
    'intra_node': 600         # Within a node (NPU to NPU-switch)
}

# Latency: 200ns for intra-node, 500ns for other links
# 200 ns = 0.2 us = 0.0002 ms
# 500 ns = 0.5 us = 0.0005 ms
latency_config = {
    'host_edge': 0.0005,
    'edge_agg': 0.0005,
    'agg_core': 0.0005,
    'intra_node': 0.0002
}

# Create an instance of the FoldedClos topology
config = {
    'K': 4,
    'bandwidth_config': bw_config,
    'latency_config': latency_config,
    'bw_unit': 'GB/s',
    'lat_unit': 'ms',
    'nodes_per_server': 8,
    'npus_per_node': 8,
    'intra_node_topology': 'switch',
    'num_nvswitches': 1
}

fc_8gpu = FoldedClos(
    K=4,
    bandwidth_config=bw_config,
    latency_config=latency_config,
    nodes_per_server=8,
    npus_per_node=8,
    intra_node_topology='switch',
    num_nvswitches=1
)

print("Folded-Clos Configuration (8 GPUs/Node):")
print(f"  - K: {fc_8gpu.K}")
print(f"  - Total servers: {fc_8gpu.total_num_servers}")
print(f"  - Nodes per server: {fc_8gpu.nodes_per_server}")
print(f"  - Total nodes: {fc_8gpu.total_num_nodes}")
print(f"  - NPUs per node: {fc_8gpu.npus_per_node}")
print(f"  - Total NPUs: {fc_8gpu.total_num_hosts}")

# Visualize the topology
fig, ax, G = visualize_topology(
    fc_8gpu, 
    "Folded-Clos (K=4) with 8 GPUs/Node (128 NPUs)", 
    figsize=(20, 10), 
    show_bw=True
)
plt.show()

# Generate topology files
# generate_topology_files('FoldedClos', 'ECMP', config, '.', 'FoldedClos_8GPU_ECMP')
# generate_topology_files('FoldedClos', 'Uniform', config, '.', 'FoldedClos_8GPU_Uniform')
generate_topology_files('FoldedClos', 'Random', config, '.', 'FoldedClos_8GPU_Det_v1')


## 4. LinksOnly Mode — Nodes and Links Without Pre-computed Paths

`LinksOnly` writes only the graph structure (nodes and links with bandwidths and latencies) to the output files, with an **empty paths section**.  
Routes are **not** pre-computed; instead, the network simulator (`network.py`) generates all ECMP shortest paths on-the-fly at runtime using NetworkX whenever a new src-dst pair is first encountered.  

Use this mode when:
- You want the simulator to perform dynamic ECMP routing (e.g., for random path selection per flow).
- Pre-computing all O(N²) paths would be too slow or memory-intensive for large topologies.


In [ ]:
bw_config = {
    'host_edge': 25,
    'edge_agg': 25,
    'agg_core': 25,
    'intra_node': 100
}

latency_config = {
    'host_edge': 0.0005,
    'edge_agg': 0.0005,
    'agg_core': 0.0005,
    'intra_node': 0.0002
}

config = {
    'K': 4,
    'bandwidth_config': bw_config,
    'latency_config': latency_config,
    'bw_unit': 'GB/s',
    'lat_unit': 'ms',
    'nodes_per_server': 8,
    'npus_per_node': 8,
    'intra_node_topology': 'switch',
    'num_nvswitches': 1
}

# 'LinksOnly' — writes only edges (nodes + links); no paths section.
# network.py will compute ECMP paths dynamically via NetworkX.
generate_topology_files('FoldedClos', 'LinksOnly', config, '.', 'FoldedClos_8GPU_LinksOnly')
